***Step 1: Tạo SparkSession và load các dataset cần thiết***

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IntrusionDetection") \
    .master("local[2]") \
    .getOrCreate()

spark

#Tập training
df_training = spark.read.csv("/home/jovyan/intrusion_data/raw/CSV Files/Training and Testing Sets/UNSW_NB15_training-set.csv",
    header=True, 
    inferSchema=True)
df_training.count()

#

175341

**-Tập ground truth**: Chứa các thông tin về các cuộc tấn công

In [13]:
df_gt = spark.read.csv("/home/jovyan/intrusion_data/raw/CSV Files/NUSW-NB15_GT.csv",
    header=True,
    inferSchema=True)
df_gt.show(5)
df_gt.columns

+----------+----------+---------------+-------------------+--------+------------+-----------+--------------+----------------+--------------------+--------------------+---+
|Start time| Last time|Attack category| Attack subcategory|Protocol|   Source IP|Source Port|Destination IP|Destination Port|         Attack Name|    Attack Reference|  .|
+----------+----------+---------------+-------------------+--------+------------+-----------+--------------+----------------+--------------------+--------------------+---+
|1421927414|1421927416| Reconnaissance|               HTTP|     tcp|175.45.176.0|      13284|149.171.126.16|              80|Domino Web Server...|                   -|  .|
|1421927415|1421927415|       Exploits|   Unix 'r' Service|     udp|175.45.176.3|      21223|149.171.126.18|           32780|Solaris rwalld Fo...|CVE 2002-0573 (ht...|  .|
|1421927416|1421927416|       Exploits|            Browser|     tcp|175.45.176.2|      23357|149.171.126.16|              80|Windows Metafil

['Start time',
 'Last time',
 'Attack category',
 'Attack subcategory',
 'Protocol',
 'Source IP',
 'Source Port',
 'Destination IP',
 'Destination Port',
 'Attack Name',
 'Attack Reference',
 '.']

**-Đọc các features của datasets:** Bao gồm 49 features khác nhau cho các lượt truy cập mạng

In [2]:
import pandas as pd
features = pd.read_csv("/home/jovyan/intrusion_data/raw/CSV Files/NUSW-NB15_features.csv", encoding="cp1252")
col_names = list(features["Name"])
print(col_names)
print(len(col_names))

['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label']
49


**-Đọc các bản ghi gốc và kiểm tra các cột:**

In [15]:
raw_df = spark.read.csv(
    "/home/jovyan/intrusion_data/raw/CSV Files/UNSW-NB15_1.csv",
    header=True,
    inferSchema=True,
    schema=None
).toDF(*col_names)

raw_df.select("srcip", "dstip", "Stime", "attack_cat").show(5)
# see the shape
print(f"Rows: {raw_df.count()}")
print(f"Columns: {len(raw_df.columns)}")

+----------+-------------+----------+----------+
|     srcip|        dstip|     Stime|attack_cat|
+----------+-------------+----------+----------+
|59.166.0.0|149.171.126.9|1421927414|      NULL|
|59.166.0.6|149.171.126.7|1421927414|      NULL|
|59.166.0.5|149.171.126.5|1421927414|      NULL|
|59.166.0.3|149.171.126.0|1421927414|      NULL|
|59.166.0.0|149.171.126.9|1421927414|      NULL|
+----------+-------------+----------+----------+
only showing top 5 rows

Rows: 700000
Columns: 49


**Kiểm tra list events:**


In [17]:
list_event_df = spark.read.csv("/home/jovyan/intrusion_data/raw/CSV Files/UNSW-NB15_LIST_EVENTS.csv",
    header=True,
    inferSchema=True,
    schema=None)

list_event_df.show()

+----------------+--------------------+----------------+
| Attack category|  Attack subcategory|Number of events|
+----------------+--------------------+----------------+
|          normal|                NULL|         2218761|
|        Fuzzers |                 FTP|             558|
|        Fuzzers |                HTTP|            1497|
|        Fuzzers |                 RIP|            3550|
|        Fuzzers |                 SMB|            5245|
|        Fuzzers |              Syslog|            1851|
|        Fuzzers |                PPTP|            1583|
|         Fuzzers|                 FTP|             248|
|         Fuzzers|              DCERPC|             164|
|         Fuzzers|                OSPF|             993|
|        Fuzzers |                TFTP|             193|
|        Fuzzers |             DCERPC |             455|
|        Fuzzers |                OSPF|            1746|
|        Fuzzers |                 BGP|            6163|
| Reconnaissance |             

**Step 2: Load các công cụ cần thiết để khám phá dữ liệu:**

In [ ]:
from pyspark.sql.functions import from_unixtime, col,to_timestamp, isnan, when, count, window

In [ ]:

raw_df.groupBy(
    window("timestamp", "5 minutes")
).agg(
    count("*").alias("packet_count")
).orderBy("window").show(10)

+--------------------+------------+
|              window|packet_count|
+--------------------+------------+
|{2015-01-22 11:45...|           6|
|{2015-01-22 11:50...|        7383|
|{2015-01-22 11:55...|        7424|
|{2015-01-22 12:00...|        8712|
|{2015-01-22 12:05...|        7805|
|{2015-01-22 12:10...|        7575|
|{2015-01-22 12:15...|        6571|
|{2015-01-22 12:20...|        6796|
|{2015-01-22 12:25...|        7506|
|{2015-01-22 12:30...|        7619|
+--------------------+------------+
only showing top 10 rows



In [ ]:


raw_df = raw_df.withColumn(
    "timestamp", 
    from_unixtime(col("Stime"))
)

raw_df.select("srcip", "timestamp", "attack_cat").show(5)
raw_df = raw_df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"))
)

raw_df.printSchema()

+----------+-------------------+----------+
|     srcip|          timestamp|attack_cat|
+----------+-------------------+----------+
|59.166.0.0|2015-01-22 11:50:14|      NULL|
|59.166.0.0|2015-01-22 11:50:14|      NULL|
|59.166.0.6|2015-01-22 11:50:14|      NULL|
|59.166.0.5|2015-01-22 11:50:14|      NULL|
|59.166.0.3|2015-01-22 11:50:14|      NULL|
+----------+-------------------+----------+
only showing top 5 rows

root
 |-- srcip: string (nullable = true)
 |-- sport: string (nullable = true)
 |-- dstip: string (nullable = true)
 |-- dsport: string (nullable = true)
 |-- proto: string (nullable = true)
 |-- state: string (nullable = true)
 |-- dur: double (nullable = true)
 |-- sbytes: integer (nullable = true)
 |-- dbytes: integer (nullable = true)
 |-- sttl: integer (nullable = true)
 |-- dttl: integer (nullable = true)
 |-- sloss: integer (nullable = true)
 |-- dloss: integer (nullable = true)
 |-- service: string (nullable = true)
 |-- Sload: double (nullable = true)
 |-- Dload: 

+--------------+-----+
|    attack_cat|count|
+--------------+-----+
|         Worms|  130|
|     Shellcode| 1133|
|       Fuzzers|18184|
|      Analysis| 2000|
|           DoS|12264|
|Reconnaissance|10491|
|      Backdoor| 1746|
|      Exploits|33393|
|        Normal|56000|
|       Generic|40000|
+--------------+-----+



+---+---+-----+-------+-----+-----+-----+------+------+----+----+----+-----+-----+-----+-----+------+------+----+----+----+-----+-----+----+------+------+------+-----+-----+-----------+-----------------+----------+------------+----------+----------------+----------------+--------------+------------+----------+----------------+----------+----------+---------------+----------+-----+
| id|dur|proto|service|state|spkts|dpkts|sbytes|dbytes|rate|sttl|dttl|sload|dload|sloss|dloss|sinpkt|dinpkt|sjit|djit|swin|stcpb|dtcpb|dwin|tcprtt|synack|ackdat|smean|dmean|trans_depth|response_body_len|ct_srv_src|ct_state_ttl|ct_dst_ltm|ct_src_dport_ltm|ct_dst_sport_ltm|ct_dst_src_ltm|is_ftp_login|ct_ftp_cmd|ct_flw_http_mthd|ct_src_ltm|ct_srv_dst|is_sm_ips_ports|attack_cat|label|
+---+---+-----+-------+-----+-----+-----+------+------+----+----+----+-----+-----+-----+-----+------+------+----+----+----+-----+-----+----+------+------+------+-----+-----+-----------+-----------------+----------+------------+-----

+---+--------+-----+-------+-----+-----+-----+------+------+---------+----+----+-----------+-----------+-----+-----+----------+----------+-----------+-----------+----+----------+----------+----+--------+--------+--------+-----+-----+-----------+-----------------+----------+------------+----------+----------------+----------------+--------------+------------+----------+----------------+----------+----------+---------------+----------+-----+
| id|     dur|proto|service|state|spkts|dpkts|sbytes|dbytes|     rate|sttl|dttl|      sload|      dload|sloss|dloss|    sinpkt|    dinpkt|       sjit|       djit|swin|     stcpb|     dtcpb|dwin|  tcprtt|  synack|  ackdat|smean|dmean|trans_depth|response_body_len|ct_srv_src|ct_state_ttl|ct_dst_ltm|ct_src_dport_ltm|ct_dst_sport_ltm|ct_dst_src_ltm|is_ftp_login|ct_ftp_cmd|ct_flw_http_mthd|ct_src_ltm|ct_srv_dst|is_sm_ips_ports|attack_cat|label|
+---+--------+-----+-------+-----+-----+-----+------+------+---------+----+----+-----------+-----------+-----+--

[('id', 'int'),
 ('dur', 'double'),
 ('proto', 'string'),
 ('service', 'string'),
 ('state', 'string'),
 ('spkts', 'int'),
 ('dpkts', 'int'),
 ('sbytes', 'int'),
 ('dbytes', 'int'),
 ('rate', 'double'),
 ('sttl', 'int'),
 ('dttl', 'int'),
 ('sload', 'double'),
 ('dload', 'double'),
 ('sloss', 'int'),
 ('dloss', 'int'),
 ('sinpkt', 'double'),
 ('dinpkt', 'double'),
 ('sjit', 'double'),
 ('djit', 'double'),
 ('swin', 'int'),
 ('stcpb', 'bigint'),
 ('dtcpb', 'bigint'),
 ('dwin', 'int'),
 ('tcprtt', 'double'),
 ('synack', 'double'),
 ('ackdat', 'double'),
 ('smean', 'int'),
 ('dmean', 'int'),
 ('trans_depth', 'int'),
 ('response_body_len', 'int'),
 ('ct_srv_src', 'int'),
 ('ct_state_ttl', 'int'),
 ('ct_dst_ltm', 'int'),
 ('ct_src_dport_ltm', 'int'),
 ('ct_dst_sport_ltm', 'int'),
 ('ct_dst_src_ltm', 'int'),
 ('is_ftp_login', 'int'),
 ('ct_ftp_cmd', 'int'),
 ('ct_flw_http_mthd', 'int'),
 ('ct_src_ltm', 'int'),
 ('ct_srv_dst', 'int'),
 ('is_sm_ips_ports', 'int'),
 ('attack_cat', 'string'),
 (

+--------+-----+
| service|count|
+--------+-----+
|       -|94168|
|     dns|47294|
|    http|18724|
|    smtp| 5058|
|ftp-data| 3995|
|     ftp| 3428|
|     ssh| 1302|
|    pop3| 1105|
|    dhcp|   94|
|    snmp|   80|
|     ssl|   56|
|     irc|   25|
|  radius|   12|
+--------+-----+



In [ ]:
df_gt.select('Start')
gt_renamed = df_gt.withColumnRenamed("Source IP", "srcip") \
                  .withColumnRenamed("Source Port", "sport") \
                  .withColumnRenamed("Destination IP", "dstip") \
                  .withColumnRenamed("Destination Port", "dsport")\
                  .withColumnRenamed("Protocol", "proto") \
                  .withColumnRenamed("Start time", "Stime")

In [ ]:
gt_renamed.printSchema()